In [ ]:
#时间序列分解

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

class MovingAvg(nn.Module):

    def __init__(self, kernel_size, stride):
        super(MovingAvg, self).__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=stride, padding=0)

    def forward(self, x):
        batch_size, _, seq_len = x.shape

        # padding on both ends of the time series
        front = x[:, :, 0:1].repeat(1, 1, (self.kernel_size - 1) // 2)
        end = x[:, :, -1:].repeat(1, 1, (self.kernel_size - 1) // 2)
        x = torch.cat([front, x, end], dim=2)  # 在序列维度进行填充

        # Apply moving average
        x = self.avg(x)  

        return x

class SeriesDecomp(nn.Module):

    def __init__(self, kernel_size):
        super(SeriesDecomp, self).__init__()
        self.moving_avg = MovingAvg(kernel_size, stride=1)

    def forward(self, x):
        decompose = []
        moving_mean = self.moving_avg(x)  # Trend component, shape [batch_size, n, seq_len]
        res = x - moving_mean  # Seasonal component, shape [batch_size, n, seq_len]
        
        decomposed = [res,moving_mean]  # 先季节性，再趋势性

        return decomposed  

In [ ]:
class MultiScaleSeasonMixing(nn.Module):
    """
    Bottom-up mixing season pattern
    """

    def __init__(self, seq_len=96, down_sampling_window=2, down_sampling_layers=2):
        super(MultiScaleSeasonMixing, self).__init__()
        self.seq_len = seq_len
        self.down_sampling_window = down_sampling_window
        self.down_sampling_layers = down_sampling_layers

        self.down_sampling_layers = torch.nn.ModuleList(
            [
                nn.Sequential(
                    torch.nn.Linear(
                        self.seq_len // (self.down_sampling_window ** i),
                        self.seq_len // (self.down_sampling_window ** (i + 1)),
                    ),
                    nn.GELU(),
                    torch.nn.Linear(
                        self.seq_len // (self.down_sampling_window ** (i + 1)),
                        self.seq_len // (self.down_sampling_window ** (i + 1)),
                    ),

                )
                for i in range(self.down_sampling_layers)
            ]
        )

    def forward(self, season_list):
        
        # mixing high->low
        season_list_reverse = season_list.copy()
        season_list_reverse.reverse()
        out_high = season_list_reverse[0]
        out_low = season_list_reverse[1]
        out_season_list = [out_high]

        for i in range(len(season_list_reverse) - 1):
            out_low_res = self.down_sampling_layers[i](out_high)
            out_low = out_low + out_low_res
            out_high = out_low
            if i + 2 <= len(season_list_reverse) - 1:
                out_low = season_list_reverse[i + 2]
            out_season_list.append(out_high)
            
        out_season_list.reverse()
        return out_season_list


class MultiScaleTrendMixing(nn.Module):
    """
    Top-down mixing trend pattern
    """

    def __init__(self, seq_len=96, down_sampling_window=2, down_sampling_layers=2):
        super(MultiScaleTrendMixing, self).__init__()
        self.seq_len = seq_len
        self.down_sampling_window = down_sampling_window
        self.down_sampling_layers = down_sampling_layers

        self.up_sampling_layers = torch.nn.ModuleList(
            [
                nn.Sequential(
                    torch.nn.Linear(
                        self.seq_len // (self.down_sampling_window ** (i + 1)),
                        self.seq_len // (self.down_sampling_window ** i),
                    ),
                    nn.GELU(),
                    torch.nn.Linear(
                        self.seq_len // (self.down_sampling_window ** i),
                        self.seq_len // (self.down_sampling_window ** i),
                    ),
                )
                for i in reversed(range(self.down_sampling_layers))
            ])

    def forward(self, trend_list):

        # mixing low->high
        out_low = trend_list[0]
        out_high = trend_list[1]
        out_trend_list = [out_low]

        for i in range(len(trend_list) - 1):
            out_high_res = self.up_sampling_layers[i](out_low)
            out_high = out_high + out_high_res
            out_low = out_high
            if i + 2 <= len(trend_list) - 1:
                out_high = trend_list[i + 2]
            out_trend_list.append(out_low)

        return out_trend_list

In [ ]:
class PHI(nn.Module): #Past Histroy Information Aggregation
    def __init__(self, seq_len=96, k=2, c=2, layers=5): #k：downsample_layer；c：downsample_window
        super(PHI, self).__init__()
        self.seq_len = seq_len
        self.k = k
        self.layers = layers
        
        # 计算 k_list 和 avg_pools
        if self.k > 0:
            self.k_list = [c ** i for i in range(k, 0, -1)]
            self.avg_pools = nn.ModuleList([nn.AvgPool1d(kernel_size=k, stride=k) for k in self.k_list])
            
            # 构建线性层
            self.linears = nn.ModuleList(
                [
                    nn.Sequential(
                        nn.Linear(self.seq_len // k, self.seq_len // k),
                        nn.GELU(),
                        nn.Linear(self.seq_len // k, self.seq_len * c // k),
                    )
                    for k in self.k_list
                ]
            )
        
        # 序列分解
        self.decomp = SeriesDecomp(kernel_size=25)
        # 季节性混合
        self.mixing_multi_scale_season = MultiScaleSeasonMixing()
        # 趋势性混合
        self.mixing_multi_scale_trend = MultiScaleTrendMixing()

    def forward(self, x):

        if self.k == 0:
            return x
        
        # 存储经过不同池化处理的结果
        sample_x = []
        for i, k in enumerate(self.k_list):
            sample_x.append(self.avg_pools[i](x))
        sample_x.append(x)
        final_list = sample_x
        
        for i in range(self.layers):
            output_list = []
            for x in final_list:
                dec = self.decomp(x)
                output_list.append(dec)

            season_list = [output[0] for output in output_list]
            trend_list = [output[1] for output in output_list]

            out_season_list = self.mixing_multi_scale_season(season_list)
            out_trend_list = self.mixing_multi_scale_trend(trend_list)
            final_list = [s + t for s, t in zip(out_season_list, out_trend_list)]
        
        n = len(final_list)
        # 多尺度融合
        for i in range(n - 1):
            tmp = self.linears[i](final_list[i])
            final_list[i + 1] = torch.add(final_list[i + 1], tmp, alpha=1.0)
            
        output = final_list[n - 1]
#         print(output.shape)
        return output
        

In [ ]:
#时间特征序列与文本原型对齐

In [ ]:
import torch
import torch.nn as nn
from torch.nn import ReplicationPad1d

class TokenEmbedding(nn.Module):
    def __init__(self, c_in, d_model):
        super(TokenEmbedding, self).__init__()
        padding = 1 if torch.__version__ >= '1.5.0' else 2
        self.tokenConv = nn.Conv1d(in_channels=c_in, out_channels=d_model,
                                   kernel_size=3, padding=padding, padding_mode='circular', bias=False)
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(
                    m.weight, mode='fan_in', nonlinearity='leaky_relu')

    def forward(self, x):
        x = self.tokenConv(x.permute(0, 2, 1)).transpose(1, 2) #nn.Conv1d改变的是第二个维度
        return x

class PatchEmbedding(nn.Module):
    def __init__(self, d_model, patch_len, stride, dropout):
        super(PatchEmbedding, self).__init__()
        # Patching
        self.patch_len = patch_len
        self.stride = stride
        self.padding_patch_layer = ReplicationPad1d((0, stride))

        # Backbone, Input encoding: projection of feature vectors onto a d-dim vector space
        self.value_embedding = TokenEmbedding(patch_len, d_model)

        # Positional embedding
        # self.position_embedding = PositionalEmbedding(d_model)

        # Residual dropout
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        n_vars = x.shape[1]
        
        x = self.padding_patch_layer(x)
        x = x.unfold(dimension=-1, size=self.patch_len, step=self.stride)
        x = torch.reshape(x, (x.shape[0] * x.shape[1], x.shape[2], x.shape[3])) #x.shape[2]：块的个数； x.shape[3]：块的长度
        # Input encoding
        x = self.value_embedding(x)
        return self.dropout(x), n_vars

In [ ]:
# multi-Attention
from math import sqrt

class ReprogrammingLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_keys=None, d_llm=None, attention_dropout=0.1):
        super(ReprogrammingLayer, self).__init__()

        d_keys = d_keys or (d_model // n_heads)

        self.query_projection = nn.Linear(d_model, d_keys * n_heads)
        self.key_projection = nn.Linear(d_llm, d_keys * n_heads)
        self.value_projection = nn.Linear(d_llm, d_keys * n_heads)
        self.out_projection = nn.Linear(d_keys * n_heads, d_llm)

        self.n_heads = n_heads
        self.dropout = nn.Dropout(attention_dropout)

    def forward(self, target_embedding, source_embedding, value_embedding):
        B, L, _ = target_embedding.shape
        S, _ = source_embedding.shape
        H = self.n_heads

        target_embedding = self.query_projection(target_embedding).view(B, L, H, -1)
        source_embedding = self.key_projection(source_embedding).view(S, H, -1)
        value_embedding = self.value_projection(value_embedding).view(S, H, -1)

        out = self.reprogramming(target_embedding, source_embedding, value_embedding)
        out = out.reshape(B, L, -1)

        return self.out_projection(out)

    def reprogramming(self, target_embedding, source_embedding, value_embedding):
        B, L, H, E = target_embedding.shape

        scale = 1. / sqrt(E)

        scores = torch.einsum("blhe,she->bhls", target_embedding, source_embedding)

        A = self.dropout(torch.softmax(scale * scores, dim=-1))
        reprogramming_embedding = torch.einsum("bhls,she->blhe", A, value_embedding)

        return reprogramming_embedding

In [ ]:
class FlattenHead(nn.Module):
    def __init__(self, patch_nums, d_ff, target_window, head_dropout=0):
        super().__init__()
        self.flatten = nn.Flatten(start_dim=-2)  # 展平 patch_nums 和 d_ff
        self.linear = nn.Linear(patch_nums * d_ff, target_window)  
        self.dropout = nn.Dropout(head_dropout)

    def forward(self, x):
        x = self.flatten(x)  # (batch_size, n, patch_nums * d_ff)
        x = self.linear(x)   # (batch_size, n, target_window)
        x = self.dropout(x)  # (batch_size, n, target_window)
        return x

In [ ]:
from transformers import BertConfig, BertModel, BertTokenizer, LlamaConfig, LlamaModel, LlamaTokenizer, GPT2Config, GPT2Model, GPT2Tokenizer
import numpy as np
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

class llmodel(nn.Module):
    def __init__(self, patch_len=16, stride=8, pred_len=12, seq_len=96):
        super(llmodel, self).__init__()
        self.d_model = 16 #注意力机制时间块输入维度
        self.patch_len = patch_len # 块长度
        self.stride = stride # patch步长
        self.pred_len = pred_len 
        self.seq_len = seq_len
        
        self.bert_config = BertConfig.from_pretrained('./LLM/BERT')
        self.bert_config.num_hidden_layers = 32
        self.bert_config.output_attentions = True
        self.bert_config.output_hidden_states = True
        
        self.llm_model = BertModel.from_pretrained(
            './LLM/BERT',
            trust_remote_code=True,
            local_files_only=True,
            config=self.bert_config,
            device_map="auto"  # 自动选择 GPU
        )
        self.tokenizer = BertTokenizer.from_pretrained(
            './LLM/BERT',
            trust_remote_code=True,
            local_files_only=True
        )

        #use gpt2
        # self.gpt2_config = GPT2Config.from_pretrained('./LLM/GPT-2')
        # self.gpt2_config.num_hidden_layers = 32
        # self.gpt2_config.output_attentions = True
        # self.gpt2_config.output_hidden_states = True
        
        # self.llm_model = GPT2Model.from_pretrained(
        #     './LLM/GPT-2',
        #     trust_remote_code=True,
        #     local_files_only=True,
        #     config=self.gpt2_config,
        #     device_map="auto"  # 自动选择 GPU
        # )
        # self.tokenizer = GPT2Tokenizer.from_pretrained(
        #     './LLM/GPT-2',
        #     trust_remote_code=True,
        #     local_files_only=True
        # )

        #use llama
        # self.llama_config = LlamaConfig.from_pretrained('./LLM/llama-7b')
        # self.llama_config.num_hidden_layers = 32
        # self.llama_config.output_attentions = True
        # self.llama_config.output_hidden_states = True
        
        # self.llm_model = LlamaModel.from_pretrained(
        #     './LLM/llama-7b',
        #     trust_remote_code=True,
        #     local_files_only=True,
        #     config=self.llama_config,
        #     device_map="auto"  # 自动选择 GPU
        # )
        # self.tokenizer = LlamaTokenizer.from_pretrained(
        #     './LLM/llama-7b',
        #     trust_remote_code=True,
        #     local_files_only=True
        # )
        if self.tokenizer.eos_token:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        else:
            pad_token = '[PAD]'
            self.tokenizer.add_special_tokens({'pad_token': pad_token})
            self.tokenizer.pad_token = pad_token
        
        for param in self.llm_model.parameters():
            param.requires_grad = False
            
        self.word_embeddings = self.llm_model.get_input_embeddings().weight# (30522,768) 
        self.vocab_size = self.word_embeddings.shape[0] 
        self.num_tokens = 1000
        self.mapping_layer = nn.Linear(self.vocab_size, self.num_tokens)
        
        self.patch_embedding = PatchEmbedding(self.d_model, self.patch_len, self.stride, 0.1)
        
        self.d_model = 16
        self.n_heads = 8
        self.d_ff = 32 #每个头的维度
        self.d_llm = 768 #gpt2、bert的隐藏层维度是768，llama隐藏层维度4096
        self.reprogramming_layer = ReprogrammingLayer(self.d_model, self.n_heads, self.d_ff, self.d_llm)
        
        self.patch_nums = int((self.seq_len - self.patch_len) / self.stride + 2)
        self.output_projection = FlattenHead(self.patch_nums, self.d_ff, self.pred_len, 0.1)
        
    def forward(self, x_enc):
#         print(self.word_embeddings.shape)
        
        #multi-attention
        source_embeddings = self.mapping_layer(self.word_embeddings.permute(1, 0)).permute(1, 0) 
        enc_out, n_vars = self.patch_embedding(x_enc) 
        enc_out = self.reprogramming_layer(enc_out, source_embeddings, source_embeddings) 
  
        dec_out = self.llm_model(inputs_embeds=enc_out).last_hidden_state 
        dec_out = dec_out[:, :, :self.d_ff]
        dec_out = torch.reshape(dec_out, (-1, n_vars, dec_out.shape[-2], dec_out.shape[-1])) 
        dec_out = self.output_projection(dec_out)                                 
        return dec_out

In [ ]:
import torch

class StandardScaler:
    def __init__(self):
        self.mean_ = None
        self.scale_ = None

    def fit(self, X):
        """
        计算每个传感器在 batch 维度上的均值和标准差。
        输入形状: (batch_size, sensors, steps) -> 沿着 batch 计算
        """
        self.mean_ = X.mean(dim=0, keepdim=True)  # (1, sensors, step)
        self.scale_ = X.std(dim=0, keepdim=True)  # (1, sensors, step)
        return self

    def transform(self, X):
        """
        使用计算的均值和标准差进行标准化
        """
        return (X - self.mean_) / (self.scale_)  

    def inverse_transform(self, X):
        """
        反标准化，恢复原始数据
        """
        return X * (self.scale_) + self.mean_

    def fit_transform(self, X):
        """
        先 fit，再 transform
        """
        return self.fit(X).transform(X)


In [ ]:
import torch
import pandas as pd

class Model(nn.Module):
    def __init__(self, seq_len=96, pred_len=12):
        super(Model, self).__init__()
        self.phi = PHI()
        self.llmodel = llmodel()
        self.scaler = StandardScaler()
        
    def forward(self, x_input):
        
        y = self.phi(x_input)
        dec_out = self.llmodel(y)
        return dec_out

In [ ]:
import numpy as np
import pandas as pd
import os
from torch.utils.data import Dataset, DataLoader

# 加载数据
data_file = os.path.join('./data', 'PEMS03.npz')
print('data file:', data_file)
data = np.load(data_file, allow_pickle=True)
data = data['data']  
data = data[:, :, 0] 
# data = data[:, :10]

# 滑动窗口参数
seq_len = 96
pred_len = 12
total_len = seq_len + pred_len  # 108
train_ratio = 0.6
val_ratio = 0.2

# 计算数据集划分索引
num_time_steps = data.shape[0]
train_end = int(train_ratio * num_time_steps)  
val_end = int((val_ratio + train_ratio) * num_time_steps)  
print(train_end,val_end)

train_raw = data[:train_end]      
val_raw = data[train_end:val_end]  
test_raw = data[val_end:]         
# print(val_raw.shape,test_raw.shape)

def apply_sliding_window(split_data, total_len, step=1):
    """
    直接在所有传感器数据 (time_steps, sensors) 维度上滑动取窗口
    默认 step=1: 每个时间步都取一个窗口
    test_batches 需要 step=12
    返回形状: (num_samples, sensors, total_len)
    """
    num_samples = (split_data.shape[0] - total_len + 1) // step  # 计算滑动步长影响的样本数
    batches = np.stack([split_data[i:i + total_len].T for i in range(0, num_samples * step, step)], axis=0)
    return batches  # (num_samples, sensors, total_len)

# 生成训练和验证数据，步长为 1
train_batches = apply_sliding_window(train_raw, total_len, step=1)
val_batches = apply_sliding_window(val_raw, total_len, step=1)

# 生成测试数据，步长为 12
test_batches = apply_sliding_window(test_raw, total_len, step=12)
ttest_batches = test_batches

# 输出结果
print("Train batch shape:", train_batches.shape) 
print("Val batch shape:", val_batches.shape)
print("Test batch shape:", test_batches.shape)

scaler = StandardScaler()
scaler.fit(torch.tensor(train_batches, dtype=torch.float32))   

train_batches = scaler.transform(torch.tensor(train_batches, dtype=torch.float32)).numpy()
val_batches = scaler.transform(torch.tensor(val_batches, dtype=torch.float32)).numpy()
test_batches = scaler.transform(torch.tensor(test_batches, dtype=torch.float32)).numpy()

class SensorDataset(Dataset):
    def __init__(self, data_batches):
        # data_batches shape: (num_batches, 108)
        self.data = torch.tensor(data_batches, dtype=torch.float32)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # 拆分历史序列和预测序列
        seq_len = 96
        pred_len = 12
        seq_x = self.data[idx, :, :seq_len]
        seq_y = self.data[idx, :, seq_len:]
        return seq_x, seq_y
    
class SensorDataset_ini(Dataset):
    def __init__(self, data_batches, data_batches_ini):
        # data_batches shape: (num_batches, 108)
        self.data = torch.tensor(data_batches, dtype=torch.float32)
        self.data_ini = torch.tensor(data_batches_ini, dtype = torch.float32)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # 拆分历史序列和预测序列
        seq_len = 96
        pred_len = 12
        seq_x = self.data[idx, :, :seq_len]
        seq_y = self.data[idx, :, seq_len:]
        seq_y_ini = self.data_ini[idx, :, seq_len:]
        return seq_x, seq_y, seq_y_ini
    
train_dataset = SensorDataset(train_batches)
val_dataset = SensorDataset(val_batches)
test_dataset = SensorDataset_ini(test_batches, ttest_batches)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=16,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=8,
    drop_last=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=8,
    drop_last=False
)

In [ ]:
print(np.isnan(train_batches).sum())
print(np.isnan(val_batches).sum())
print(np.isnan(test_batches).sum())

In [ ]:
#--------------------------训练+测试-------------------------------

In [ ]:
#----------------mape带截断----------------------------------
def calculate_metrics(pred, true, true_ini=None):
    """
    pred: numpy array, 预测值
    true: numpy array, 真实值
    true_ini: 可选，原始（未反标准化）真实值，若不需要过滤0可以忽略
    """
    mse = np.mean((pred - true) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(pred - true))

    # 计算 MAPE（带截断）
    with np.errstate(divide='ignore', invalid='ignore'): #如果分母是0不会做警报，会平滑的过去，但是分母是0的问题（inf）依然存在
        mape = np.abs((pred - true) / true)
        mape = np.where(mape > 5, 0, mape)  
        mape = np.nanmean(mape) * 100 
                                       

    return rmse, mae, mape

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import time
import numpy as np
from accelerate import Accelerator
import random


# 设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 超参数
epochs = 20
learning_rate = 0.0001
criterion = nn.MSELoss()

# 初始化模型、优化器、调度器
model = Model(seq_len=96, pred_len=12).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate, eps=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)

# 使用 accelerate
accelerator = Accelerator(mixed_precision='fp16')  
model, optimizer, train_loader, val_loader, test_loader = accelerator.prepare(
    model, optimizer, train_loader, val_loader, test_loader
)

# 提取 scaler 的 mean 和 std
scaler_mean = scaler.mean_.cpu().numpy()
scaler_std = scaler.scale_.cpu().numpy()

pred_len = test_loader.dataset[0][1].shape[1]  
scaler_mean_last12 = scaler_mean[:, :, -pred_len:]
scaler_std_last12 = scaler_std[:, :, -pred_len:]

# 设定梯度累积的步数（例如 10）
accumulation_steps = 24 

# 训练 + 验证 + 测试
for epoch in range(epochs):
    # ========== 训练 ==========
    model.train()
    train_losses = []
    start_time = time.time()

    optimizer.zero_grad()  # 确保梯度清零
    for batch_idx, (seq_x, seq_y) in enumerate(train_loader):
        seq_x, seq_y = seq_x.float().to(device), seq_y.float().to(device)

        with accelerator.autocast():
            output = model(seq_x)
            loss = criterion(output, seq_y) / accumulation_steps  # 归一化 loss

        accelerator.backward(loss)  # 反向传播（但不更新参数）
        train_losses.append(loss.item() * accumulation_steps)  # 存回原始 loss 值

        # 只有在 accumulation_steps 次后才更新参数
        if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(train_loader):
            optimizer.step()
            optimizer.zero_grad()  # 清空梯度

    avg_train_loss = np.mean(train_losses)
    print(f"[Epoch {epoch+1}] Train Loss: {avg_train_loss:.6f} | Time: {time.time() - start_time:.2f}s")


    # ========== 验证 ==========
    model.eval()
    val_losses = []
    with torch.no_grad():
        for seq_x, seq_y in val_loader:
            seq_x, seq_y = seq_x.float().to(device), seq_y.float().to(device)

            with accelerator.autocast():
                output = model(seq_x)
                loss = criterion(output, seq_y)

            val_losses.append(loss.item())

    avg_val_loss = np.mean(val_losses)
    print(f"[Epoch {epoch+1}] Validation Loss: {avg_val_loss:.6f}")

    # 更新学习率
    scheduler.step(avg_val_loss)

    # ========== 测试 ==========
    all_rmse, all_mae, all_mape = [], [], []  
    with torch.no_grad():
        for seq_x, seq_y, _ in test_loader:
            seq_x, seq_y = seq_x.float().to(device), seq_y.float().to(device)

            with accelerator.autocast():
                output = model(seq_x)

            pred = output.cpu().numpy()
            true = seq_y.cpu().numpy()

            pred_denorm = pred * scaler_std_last12 + scaler_mean_last12  
            true_denorm = true * scaler_std_last12 + scaler_mean_last12  

            rmse, mae, mape = calculate_metrics(pred_denorm, true_denorm)
            all_rmse.append(rmse)
            all_mae.append(mae)
            all_mape.append(mape)

    avg_rmse, avg_mae, avg_mape = np.mean(all_rmse), np.mean(all_mae), np.mean(all_mape)

    print(f"📊 [Epoch {epoch+1}] Test Metrics:")
    print(f"   🔵 MAE: {avg_mae:.4f}")
    print(f"   🟣 MAPE: {avg_mape:.2f}%")
    print(f"   🟢 RMSE: {avg_rmse:.4f}")
    
        # 保存模型
    torch.save(model.state_dict(), f"./WeightFile/model_epoch_{epoch+1}.pth")  
    print(f"💾 Model saved at epoch {epoch+1}")

    # 清理缓存
    torch.cuda.empty_cache()

print("Training & Testing Completed ✅")